# Qwen2.5-Coder 3B on Kaggle Notebooks

This is an optional notebook-local inference fallback for the Colab workflow. It does not expose a network service or connect to the workspace task board. Kaggle accelerator availability, weekly usage allowances, session limits, and storage policies can change; check the current Kaggle account and notebook UI rather than relying on a fixed GPU-hours figure.

**Recommended setup:** Create a Kaggle Dataset containing the Hugging Face snapshot for `Qwen/Qwen2.5-Coder-3B-Instruct`, then attach it with **Add Input**. Select one GPU (T4 or P100 when available) and enable Internet if dependencies still need installation. This avoids downloading the multi-gigabyte model during each inference session. Do not select two GPUs for this 3B 4-bit model; the measured T4 run used about 2 GiB peak allocated VRAM, and this notebook explicitly places the model on the first GPU.

The attached dataset is read-only and persists independently of the session. `/kaggle/working` is session output storage, not a substitute for an attached persistent model dataset. Start a session only when ready, run setup and inference as needed, and use Kaggle's **Stop Session** control when finished. Do not expect a notebook background commit to preserve a running interactive GPU session.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU accelerator and start a new session before loading the model.")
for index in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {gpu.name} ({gpu.total_memory / 1024**3:.1f} GiB VRAM)")
print(f"CUDA: {torch.version.cuda}")

## Attach the model dataset

Use **Add Input** to attach the dataset containing the model snapshot. The path cell first checks the expected dataset path, then searches attached inputs for the Qwen2.5-Coder-3B config; it errors if there are no matches or multiple matches. The mounted input is read-only: Hugging Face and pip caches are directed to writable `/kaggle/working` before Transformers is imported. Model loading is local-only and never downloads into the attached dataset.

In [ ]:
import json
import os
from pathlib import Path

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
INPUT_ROOT = Path("/kaggle/input")
PREFERRED_MODEL_DIR = INPUT_ROOT / "datasets/edmundrosariosunil/qwen2-5-coder-3b-instruct/qwen2.5-coder-3b-instruct"
HF_HOME = Path("/kaggle/working/hf_cache")
PIP_CACHE_DIR = Path("/kaggle/working/pip-cache")
HF_HOME.mkdir(parents=True, exist_ok=True)
PIP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE_DIR)
def is_target_model_config(config):
    repo_hint = str(config.get("_name_or_path", "")).lower()
    architectures = config.get("architectures", [])
    matching_model_shape = config.get("model_type") == "qwen2" and config.get("hidden_size") == 2048 and config.get("num_hidden_layers") == 36
    return "qwen2.5-coder-3b-instruct" in repo_hint or (matching_model_shape and "Qwen2ForCausalLM" in architectures)
preferred_config_path = PREFERRED_MODEL_DIR / "config.json"
if preferred_config_path.is_file() and is_target_model_config(json.loads(preferred_config_path.read_text(encoding="utf-8"))):
    MODEL_DIR = PREFERRED_MODEL_DIR
else:
    matching_dirs = []
    for config_path in INPUT_ROOT.rglob("config.json"):
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            continue
        if is_target_model_config(config):
            matching_dirs.append(config_path.parent)
    matching_dirs = sorted(set(matching_dirs))
    if len(matching_dirs) != 1:
        raise FileNotFoundError(f"Expected exactly one attached {MODEL_ID} snapshot; found {matching_dirs}. Attach its Kaggle Dataset or set PREFERRED_MODEL_DIR to the directory containing config.json.")
    MODEL_DIR = matching_dirs[0]
if not list(MODEL_DIR.glob("*.safetensors")):
    raise FileNotFoundError(f"No safetensors model weights found in {MODEL_DIR}; check the attached dataset directory.")
print(f"Model snapshot: {MODEL_DIR}")
print(f"Writable Hugging Face cache: {HF_HOME}; pip cache: {PIP_CACHE_DIR}")

## Install inference dependencies

Kaggle's preinstalled PyTorch is reused. This upgrades and force-reinstalls the selected Hugging Face and quantization packages from the pinned requirements. The pip cache under `/kaggle/working` is not assumed to survive a new session. Run this cell if the installed Hub/Transformers versions are incompatible; otherwise it can be skipped to save setup time.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "--cache-dir", str(PIP_CACHE_DIR), "huggingface_hub>=0.25.0,<1.0", "transformers>=4.45,<5", "accelerate", "bitsandbytes"])
print(f"Using PyTorch {torch.__version__}.")

## Load the model and run inference

The model uses 4-bit NF4 weights, FP16 compute, SDPA attention, and the KV cache. `device_map={"": 0}` keeps it on the first GPU even when a multi-GPU accelerator is selected; one GPU is sufficient for this model based on the prior T4 measurement. The one-token warm-up keeps the benchmark comparable with the Colab notebook and isolates steady-state generation speed; it is not a reduction in total work for a single request.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True, cache_dir=str(HF_HOME / "hub"))
quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(str(MODEL_DIR), quantization_config=quantization, device_map={"": 0}, torch_dtype=torch.float16, attn_implementation="sdpa", local_files_only=True, cache_dir=str(HF_HOME / "hub"))
model.eval()
print(f"Loaded {MODEL_ID} in 4-bit mode on GPU 0 ({torch.cuda.get_device_name(0)}).")

In [ ]:
import json
import time

task = {"title": "Add validation to the task creation API", "project": "api", "task_type": "feature", "description": "Reject invalid task input and return clear errors.", "acceptance_criteria": "Invalid titles and unsupported fields return 400; valid tasks still persist."}
messages = [{"role": "system", "content": "Refine this software task. Return only a JSON object with string fields description and acceptance_criteria. The acceptance_criteria value must be one string, never an array or object. Preserve intent; do not invent requirements or claim work is done. Make criteria concise and verifiable."}, {"role": "user", "content": json.dumps(task, ensure_ascii=False)}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda:0")
if inputs["input_ids"].shape[-1] > 2048: raise ValueError("Prompt exceeds the 2048-token evaluation budget; shorten the task fields.")
with torch.inference_mode(): warmup_output = model.generate(**inputs, max_new_tokens=1, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()
del warmup_output
torch.cuda.reset_peak_memory_stats(0)
started = time.perf_counter()
with torch.inference_mode(): output = model.generate(**inputs, max_new_tokens=220, do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
torch.cuda.synchronize()
elapsed = time.perf_counter() - started
peak_gpu_memory_mb = torch.cuda.max_memory_allocated(0) / 1024**2
peak_gpu_reserved_mb = torch.cuda.max_memory_reserved(0) / 1024**2
generated_tokens = output.shape[-1] - inputs["input_ids"].shape[-1]
answer = tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
print(f"Elapsed: {elapsed:.2f}s | Generated tokens: {generated_tokens} | Speed: {generated_tokens / max(elapsed, 1e-6):.2f} tokens/s | Peak allocated VRAM: {peak_gpu_memory_mb:.0f} MiB | Peak reserved VRAM: {peak_gpu_reserved_mb:.0f} MiB")
print(answer)

In [ ]:
try:
    draft = None
    json_answer = answer.strip()
    if json_answer.startswith("```"):
        json_answer = json_answer.removeprefix("```json").removeprefix("```")
        json_answer = json_answer.removesuffix("```").strip()
    draft = json.loads(json_answer)
    if not isinstance(draft, dict):
        raise ValueError("The model response must be a JSON object")
    criteria = draft.get("acceptance_criteria")
    if isinstance(criteria, list):
        formatted = []
        for index, item in enumerate(criteria, start=1):
            if isinstance(item, str):
                details = item.strip()
            elif isinstance(item, dict) and all(isinstance(value, str) for value in item.values()):
                details = "; ".join(f"{field.replace('_', ' ')}: {value.strip()}" for field, value in item.items() if value.strip())
            else:
                raise ValueError("Invalid acceptance criterion")
            if not details:
                raise ValueError("Empty acceptance criterion")
            formatted.append(f"{index}. {details}")
        draft["acceptance_criteria"] = "\n".join(formatted)
    assert isinstance(draft.get("description"), str)
    assert isinstance(draft.get("acceptance_criteria"), str)
    print("Output check: valid JSON with description and acceptance_criteria.")
except (json.JSONDecodeError, AssertionError, ValueError):
    print("Output check: the model did not return the expected JSON shape; review or retry manually.")

## Stop GPU usage

For an interactive session, stop it using the notebook's **Stop Session** control when finished. This notebook does not try to stop the session from Python. For a CLI-submitted batch run, Kaggle manages the run lifecycle after execution; this cleanup only releases Python/GPU memory.

In [ ]:
from pathlib import Path
import json

if not isinstance(draft, dict) or not isinstance(draft.get("description"), str) or not isinstance(draft.get("acceptance_criteria"), str):
    raise ValueError("No validated model response is available; inspect the output validation cell.")
response_path = Path("/kaggle/working/response.json")
response_path.write_text(json.dumps(draft, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Validated response saved to {response_path}.")

In [ ]:
import gc

for name in ("inputs", "output", "model", "tokenizer", "answer", "draft"):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model objects cleared. Python cleanup does not stop an interactive Kaggle session.")